# 🧩 Báo cáo: Tiền xử lý và Chuẩn hóa dữ liệu ERA5 Việt Nam (2017–2024)

## 🎯 Mục tiêu
Bài thực hành này nhằm **hợp nhất, chuẩn hóa, và chia bộ dữ liệu ERA5 của Việt Nam** giai đoạn 2017–2024 để phục vụ cho các bài toán dự báo khí tượng (như nhiệt độ, lượng mưa, độ ẩm,...).

Các bước chính gồm:
1. Thiết lập cấu hình và danh sách biến.
2. Ghép các file `.nc` theo từng biến và giai đoạn.
3. Gộp tất cả biến thành một `xarray.Dataset` duy nhất.
4. Tiền xử lý và chuẩn hóa dữ liệu.
5. Lưu file tổng hợp đã chuẩn hóa.
6. Chia tập dữ liệu thành Train / Validation / Test.
7. Kiểm tra lại toàn bộ kết quả.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import json

# ================== 1. THIẾT LẬP CẤU HÌNH ==================
base_dir = "/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/1.1_Raw_Data"
processed_dir = "/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/1.2_Processed_Data"
splited_dir = "/content/drive/MyDrive/Rain_Forecast_Fourier_Convolutional_Transformer/1_Data/1.3_Splited_Data"

# Danh sách các biến khí tượng cần xử lý
variable_names = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "mean_sea_level_pressure",
    "surface_pressure",
    "total_precipitation",
    "surface_solar_radiation_downwards",
    "skin_temperature",
    "total_column_water_vapour",
]

# Giai đoạn dữ liệu
periods = ["2017_2020", "2021_2024"]
print("Bắt đầu quá trình gộp và chuẩn hóa dữ liệu...")

Bắt đầu quá trình gộp và chuẩn hóa dữ liệu...


## 2️⃣ Ghép các file NetCDF theo từng biến
Ở bước này, ta **mở và nối dữ liệu của từng biến khí tượng** (ví dụ: nhiệt độ, gió, mưa...) theo chiều thời gian `time`.
Việc này giúp tạo ra các dataset hoàn chỉnh cho từng biến trước khi hợp nhất chúng lại.

In [ ]:
# ================== 2. GHÉP CÁC FILE THEO TỪNG BIẾN ==================
all_variables_data = []
print("\nBước 2: Ghép các file theo từng biến...")

for var in variable_names:
    files_to_concat = [os.path.join(base_dir, f"era5_vn_{var}_{p}.nc") for p in periods]
    existing_files = [f for f in files_to_concat if os.path.exists(f)]

    if not existing_files:
        print(f"⚠️ Không tìm thấy file nào cho biến '{var}'. Bỏ qua.")
        continue

    print(f"-> Đang xử lý biến '{var}' từ {len(existing_files)} file...")
    ds_var = xr.open_mfdataset(existing_files, combine='by_coords')
    all_variables_data.append(ds_var)


Bước 2: Ghép các file theo từng biến...
-> Đang xử lý biến '2m_temperature' từ 2 file...
-> Đang xử lý biến '2m_dewpoint_temperature' từ 2 file...
-> Đang xử lý biến '10m_u_component_of_wind' từ 2 file...
-> Đang xử lý biến '10m_v_component_of_wind' từ 2 file...
-> Đang xử lý biến 'mean_sea_level_pressure' từ 2 file...
-> Đang xử lý biến 'surface_pressure' từ 2 file...
-> Đang xử lý biến 'total_precipitation' từ 2 file...
-> Đang xử lý biến 'surface_solar_radiation_downwards' từ 2 file...
-> Đang xử lý biến 'skin_temperature' từ 2 file...
-> Đang xử lý biến 'total_column_water_vapour' từ 2 file...


## 3️⃣ Gộp tất cả các biến thành một dataset duy nhất
Sau khi từng biến được nối lại, ta **gộp toàn bộ chúng** thành một dataset chung `ds_all`.
Điều này giúp cho việc chuẩn hóa và xử lý đồng bộ giữa các biến dễ dàng hơn.

In [ ]:
# ================== 3. GỘP TẤT CẢ CÁC BIẾN ==================
print("\nBước 3: Gộp tất cả các biến thành một dataset duy nhất...")
ds_all = xr.merge(all_variables_data)

# Đổi tên trục thời gian nếu cần
if 'time' in ds_all.coords:
    ds_all = ds_all.rename({'time': 'valid_time'})

print("✅ Gộp dữ liệu thành công!")
print(ds_all)


Bước 3: Gộp tất cả các biến thành một dataset duy nhất...


/tmp/ipython-input-202942428.py:3: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_all = xr.merge(all_variables_data)


✅ Gộp dữ liệu thành công!
<xarray.Dataset> Size: 3GB
Dimensions:     (valid_time: 35064, latitude: 65, longitude: 33)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 281kB 2017-01-01 ... 2024-12-31T2...
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    number      int64 8B 0
    expver      (valid_time) <U4 561kB dask.array<chunksize=(17532,), meta=np.ndarray>
Data variables:
    t2m         (valid_time, latitude, longitude) float32 301MB dask.array<chunksize=(5844, 22, 11), meta=np.ndarray>
    d2m         (valid_time, latitude, longitude) float32 301MB dask.array<chunksize=(5844, 22, 11), meta=np.ndarray>
    u10         (valid_time, latitude, longitude) float32 301MB dask.array<chunksize=(5844, 22, 11), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 301MB dask.array<chunksize=(5844, 22, 11), meta=np.ndarray>
    msl         (valid_tim

/tmp/ipython-input-202942428.py:3: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_all = xr.merge(all_variables_data)


## 4️⃣ Tiền xử lý và Chuẩn hóa dữ liệu
### 🔹 Mục tiêu:
- Đảm bảo **tất cả biến có đơn vị và giá trị nằm trong phạm vi ổn định**.
- Giảm **skewness** cho các biến lệch (như mưa), giúp mô hình học hiệu quả hơn.
- Chuẩn hóa toàn bộ dữ liệu theo **Z-score** để các biến có cùng thang đo.

### 🔹 Các bước chính:
1. Chuyển đơn vị lượng mưa từ m → mm và áp dụng `log1p` để làm trơn.
2. Áp dụng chuẩn hóa Z-score cho mọi biến.
3. Lưu lại thông tin thống kê (`mean`, `std`, `units`, `transform`) vào file JSON.

In [ ]:
# ================== 4. TIỀN XỬ LÝ VÀ CHUẨN HÓA ==================
print("\nBước 4: Bắt đầu tiền xử lý và chuẩn hóa...")
ds_norm = ds_all.copy(deep=True)

# Xử lý đặc biệt cho biến mưa
tp_var_name = None
for var in ds_norm.data_vars:
    if 'precipitation' in var or var == 'tp':
        tp_var_name = var
        break

if tp_var_name:
    print(f"-> Xử lý đặc biệt cho biến mưa: '{tp_var_name}'")
    ds_norm[tp_var_name] = ds_norm[tp_var_name] * 1000  # m -> mm
    ds_norm[tp_var_name].attrs['units'] = 'mm'
    ds_norm[tp_var_name] = np.log1p(ds_norm[tp_var_name])  # log-transform
    ds_norm[tp_var_name].attrs['transform'] = 'log1p'
else:
    print("⚠️ Không tìm thấy biến mưa để xử lý đặc biệt.")

# Áp dụng chuẩn hóa Z-score
print("\n-> Áp dụng chuẩn hóa Z-score cho tất cả các biến...")
norm_stats = {}
data_vars_list = list(ds_norm.data_vars)

for var in data_vars_list:
    original_units = ds_all[var].attrs.get('units', 'no_units')
    transform_method = 'z-score'
    if var == tp_var_name:
        transform_method = 'log1p + z-score'
        units = 'mm'
    else:
        units = original_units

    mean = float(ds_norm[var].mean().values)
    std = float(ds_norm[var].std().values)

    if std > 1e-9:
        ds_norm[var] = (ds_norm[var] - mean) / std
    else:
        print(f"   - Độ lệch chuẩn quá nhỏ cho '{var}', bỏ qua.")

    norm_stats[var] = {"mean": mean, "std": std, "units": units, "transform": transform_method}

stats_path = os.path.join(processed_dir, 'normalization_stats_2017_2024.json')
with open(stats_path, 'w') as f:
    json.dump(norm_stats, f, indent=4)
print(f"✅ Lưu thông số chuẩn hóa vào: {stats_path}")


Bước 4: Bắt đầu tiền xử lý và chuẩn hóa...
-> Xử lý đặc biệt cho biến mưa: 'tp'

-> Áp dụng chuẩn hóa Z-score cho tất cả các biến...
✅ Lưu thông số chuẩn hóa vào: /content/drive/MyDrive/era5_vn_min/processed/normalization_stats_2017_2024.json


## 5️⃣ Lưu Dataset đã chuẩn hóa
Dữ liệu sau khi được chuẩn hóa sẽ được lưu lại dưới định dạng `.nc` với **nén zlib** để tiết kiệm dung lượng.

In [ ]:
# ================== 5. LƯU FILE ==================
print("\nBước 5: Lưu dataset đã gộp và chuẩn hóa...")
encoding = {var: {'zlib': True, 'complevel': 4} for var in ds_norm.data_vars}
save_path = os.path.join(processed_dir, 'era5_vn_merged_normalized_2017_2024.nc')
ds_norm.to_netcdf(save_path, encoding=encoding)
print(f"✅ Dataset đã lưu tại: {save_path}")


Bước 5: Lưu dataset đã gộp và chuẩn hóa...
✅ Dataset đã lưu tại: /content/drive/MyDrive/era5_vn_min/cc/era5_vn_merged_normalized_2017_2024.nc


## 6️⃣ Chia dữ liệu thành Train / Validation / Test
Dữ liệu được chia theo năm để đảm bảo tính thời gian:
- **Train:** 2017–2022
- **Validation:** 2023
- **Test:** 2024

In [ ]:
# ================== 6. CHIA DỮ LIỆU ==================
print("\nBước 6: Chia dữ liệu thành các tập Train / Validation / Test...")
ds_train = ds_norm.sel(valid_time=slice('2017-01-01', '2022-12-31'))
ds_val   = ds_norm.sel(valid_time=slice('2023-01-01', '2023-12-31'))
ds_test  = ds_norm.sel(valid_time=slice('2024-01-01', '2024-12-31'))

first_var = list(ds_norm.data_vars)[0]
print(f"Train shape: {ds_train[first_var].shape}")
print(f"Val shape: {ds_val[first_var].shape}")
print(f"Test shape: {ds_test[first_var].shape}")

ds_train.to_netcdf(os.path.join(splited_dir, 'train_2017_2022.nc'), encoding=encoding)
ds_val.to_netcdf(os.path.join(splited_dir, 'val_2023.nc'), encoding=encoding)
ds_test.to_netcdf(os.path.join(splited_dir, 'test_2024.nc'), encoding=encoding)
print("✅ Đã lưu thành công các tập train, val, test.")


Bước 6: Chia dữ liệu thành các tập Train / Validation / Test...
Train shape: (26292, 65, 33)
Val shape: (4380, 65, 33)
Test shape: (4392, 65, 33)
✅ Đã lưu thành công các tập train, val, test.


## 7️⃣ Kiểm tra cuối cùng và Tổng kết
Ở bước này, ta hiển thị thông tin tổng quát của ba tập dữ liệu sau khi chia để đảm bảo quá trình xử lý không có lỗi.

In [ ]:
# ================== 7. KIỂM TRA CUỐI CÙNG ==================
print("\nBước 7: Kiểm tra thông tin các tập dữ liệu đã chia...")
print("\n--- Train Dataset ---")
print(ds_train)
print("\n--- Validation Dataset ---")
print(ds_val)
print("\n--- Test Dataset ---")
print(ds_test)
print("\n🎯 Toàn bộ quá trình đã hoàn tất!")


Bước 7: Kiểm tra thông tin các tập dữ liệu đã chia...

--- Train Dataset ---
<xarray.Dataset> Size: 2GB
Dimensions:     (valid_time: 26292, latitude: 65, longitude: 33)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 210kB 2017-01-01 ... 2022-12-31T2...
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    number      int64 8B 0
    expver      (valid_time) <U4 421kB dask.array<chunksize=(17532,), meta=np.ndarray>
Data variables:
    t2m         (valid_time, latitude, longitude) float32 226MB dask.array<chunksize=(5844, 22, 11), meta=np.ndarray>
    d2m         (valid_time, latitude, longitude) float32 226MB dask.array<chunksize=(5844, 22, 11), meta=np.ndarray>
    u10         (valid_time, latitude, longitude) float32 226MB dask.array<chunksize=(5844, 22, 11), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 226MB dask.array<chunksize=(5844, 